# Visualize a coupled AIM + ocean cs32 run

Reads the native cubed-sphere Zarr written by `datagen.cpl_aim_ocn.scripts.run`
and renders atmosphere/ocean snapshots, six-face panels, a cube-unfolded view,
coupled time series, vertical AIM sigma-level structure, and a 3D spherical
view of the cs32 grid.

Dataset layout: atmosphere/ocean variables live on `(time, face, j, i)`, with
atmospheric 3D fields additionally on a vertical sigma coordinate. The writer
usually exposes this as `Zsigma=5`, while raw MITgcm-style outputs may keep a
different vertical-dimension name. The canonical on-disk grid is
the native MITgcm cubed sphere, with `face=6, j=i=32` and cell-centre longitude
/ latitude coordinates `XC(face, j, i)` and `YC(face, j, i)`.

## Environment

Needs `xarray`, `zarr`, `numpy`, `matplotlib`, `tqdm`, and `imageio[ffmpeg]`.
These ship with the `datagen` project environment.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 - registers 3D projection
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Override directly with:
#   CPL_AIM_OCN_ZARR=/path/to/run.zarr jupyter notebook notebooks/cpl_aim_ocn_visualize.ipynb
zarr_override = os.environ.get("CPL_AIM_OCN_ZARR")
if zarr_override:
    DATA_PATH = Path(zarr_override)
else:
    DATA_ROOT = Path(os.environ.get(
        "DATA_ROOT",
        f"{os.environ.get('SCRATCH', '/tmp')}/flowers-data",
    ))
    RUN_ID = os.environ.get("RUN_ID", "0000")
    run_name = f"run_{int(RUN_ID):04d}" if RUN_ID.isdigit() else RUN_ID
    root = DATA_ROOT / "cpl_aim_ocn"
    candidates = [
        root / "runs" / run_name / "run.zarr",
        root / "train" / f"{run_name}.zarr",
        root / "val" / f"{run_name}.zarr",
        root / "test" / f"{run_name}.zarr",
    ]
    DATA_PATH = next((p for p in candidates if p.exists()), candidates[0])

ds = xr.open_zarr(DATA_PATH, consolidated=None)

def find_dataset_name(dataset, *names):
    for name in names:
        if name in dataset.coords or name in dataset.data_vars:
            return name
    available = sorted(set(dataset.coords) | set(dataset.data_vars))
    raise KeyError(f"None of {names} found in dataset. Available names: {available}")

lon_name = find_dataset_name(ds, "XC", "xc", "lon", "longitude")
lat_name = find_dataset_name(ds, "YC", "yc", "lat", "latitude")
corner_lon_name = None
corner_lat_name = None
try:
    corner_lon_name = find_dataset_name(ds, "XG", "xg")
    corner_lat_name = find_dataset_name(ds, "YG", "yg")
except KeyError:
    pass

ds

In [ ]:
print("Path:", DATA_PATH)
print("Dims:", dict(ds.sizes))
if "time" in ds:
    print("Time range:", float(ds.time.min()) / 86400, "..",
          float(ds.time.max()) / 86400, "days")
print("Lon range:", float(ds[lon_name].min()), "..", float(ds[lon_name].max()))
print("Lat range:", float(ds[lat_name].min()), "..", float(ds[lat_name].max()))
print("Atmosphere vars:", sorted(v for v in ds.data_vars if v.startswith("atm_")))
print("Ocean vars:     ", sorted(v for v in ds.data_vars if v.startswith("ocn_")))
print("Run attrs:")
for k in ["co2_ppm", "solar_scale", "solar_const_w_m2", "gm_kappa", "seed", "spinup_days", "data_days"]:
    if k in ds.attrs:
        print(f"  {k}: {ds.attrs[k]}")

## Helpers

These functions keep the notebook close to the other visualization notebooks:
simple six-face panels, a quick cube cross layout, and a lightweight spherical
plot that uses the stored cubed-sphere lon/lat coordinates.

In [ ]:
HORIZONTAL_DIMS = {"time", "face", "j", "i"}
VERTICAL_DIM_PREFERENCES = ("Zsigma", "sigma", "Z", "z", "k", "lev", "level")

def vertical_dim(arr):
    for dim in VERTICAL_DIM_PREFERENCES:
        if dim in arr.dims:
            return dim
    candidates = [dim for dim in arr.dims if dim not in HORIZONTAL_DIMS]
    if len(candidates) == 1:
        return candidates[0]
    return None

def require_vertical_dim(arr, name):
    vdim = vertical_dim(arr)
    if vdim is None:
        raise ValueError(
            f"{name} has no detected vertical dimension; dims={arr.dims}. "
            "Check that you opened a vertically resolved raw run Zarr."
        )
    return vdim

def select_vertical(arr, index=-1):
    vdim = vertical_dim(arr)
    if vdim is None:
        return arr
    return arr.isel({vdim: index})

def get_field(name, *, time=-1, sigma=None):
    arr = ds[name]
    if "time" in arr.dims:
        arr = arr.isel(time=time)
    if sigma is not None:
        vdim = require_vertical_dim(arr, name)
        arr = arr.isel({vdim: sigma})
    return arr.values

def robust_limits(arr, symmetric=False, q=(2, 98)):
    arr = np.asarray(arr)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return 0.0, 1.0
    if symmetric:
        m = float(np.nanpercentile(np.abs(finite), q[1]))
        return -m, m
    return tuple(float(x) for x in np.nanpercentile(finite, q))

def show_panel(ax, arr, title, *, cmap="viridis", vmin=None, vmax=None):
    im = ax.imshow(arr, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax,
                   aspect="equal")
    ax.set_title(title, fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
    return im

def plot_faces(field_name, *, time=-1, sigma=None, cmap="viridis", symmetric=False):
    arr = get_field(field_name, time=time, sigma=sigma)
    vmin, vmax = robust_limits(arr, symmetric=symmetric)
    fig, axes = plt.subplots(1, ds.sizes["face"], figsize=(14, 2.6), constrained_layout=True)
    for f, ax in enumerate(axes):
        im = show_panel(ax, arr[f], f"face {f + 1}", cmap=cmap, vmin=vmin, vmax=vmax)
    label = field_name if sigma is None else f"{field_name}, sigma index {sigma}"
    t_days = float(ds.time.isel(time=time)) / 86400 if "time" in ds else np.nan
    fig.suptitle(f"{label} at t = {t_days:.1f} days")
    fig.colorbar(im, ax=axes, shrink=0.75, location="right")
    return fig

def plot_cross(field_name, *, time=-1, sigma=None, cmap="viridis", symmetric=False):
    arr = get_field(field_name, time=time, sigma=sigma)
    vmin, vmax = robust_limits(arr, symmetric=symmetric)
    fig, axes = plt.subplots(3, 4, figsize=(12, 8), constrained_layout=True)
    for ax in axes.flat:
        ax.axis("off")
    placements = [(0, 0, 4), (1, 0, 0), (1, 1, 1), (1, 2, 2), (1, 3, 3), (2, 0, 5)]
    for row, col, face in placements:
        ax = axes[row, col]
        ax.axis("on")
        im = show_panel(ax, arr[face], f"face {face + 1}", cmap=cmap, vmin=vmin, vmax=vmax)
    label = field_name if sigma is None else f"{field_name}, sigma index {sigma}"
    fig.suptitle(f"Cube-cross view: {label}")
    fig.colorbar(im, ax=axes, shrink=0.75, location="right")
    return fig

def lonlat_to_xyz(lon_deg, lat_deg, radius=1.0):
    lon = np.deg2rad(lon_deg)
    lat = np.deg2rad(lat_deg)
    x = radius * np.cos(lat) * np.cos(lon)
    y = radius * np.cos(lat) * np.sin(lon)
    z = radius * np.sin(lat)
    return x, y, z

def configure_sphere_axes(ax):
    ax.set_box_aspect((1, 1, 1))
    ax.set_xlim(-1.03, 1.03)
    ax.set_ylim(-1.03, 1.03)
    ax.set_zlim(-1.03, 1.03)
    ax.set_axis_off()
    if hasattr(ax, "set_proj_type"):
        ax.set_proj_type("ortho")

def cell_corner_polygons(values, lon_corner, lat_corner):
    polygons = []
    polygon_values = []
    nface, nj, ni = values.shape
    if lon_corner.shape[0] < nface or lon_corner.shape[-2] < nj + 1 or lon_corner.shape[-1] < ni + 1:
        return polygons, polygon_values
    for face in range(nface):
        for jj in range(nj):
            for ii in range(ni):
                lon_cell = np.array([
                    lon_corner[face, jj, ii],
                    lon_corner[face, jj, ii + 1],
                    lon_corner[face, jj + 1, ii + 1],
                    lon_corner[face, jj + 1, ii],
                ])
                lat_cell = np.array([
                    lat_corner[face, jj, ii],
                    lat_corner[face, jj, ii + 1],
                    lat_corner[face, jj + 1, ii + 1],
                    lat_corner[face, jj + 1, ii],
                ])
                value = values[face, jj, ii]
                if not (np.isfinite(value) and np.all(np.isfinite(lon_cell)) and np.all(np.isfinite(lat_cell))):
                    continue
                x, y, z = lonlat_to_xyz(lon_cell, lat_cell)
                polygons.append(list(zip(x, y, z)))
                polygon_values.append(value)
    return polygons, np.asarray(polygon_values)

def plot_sphere(field_name, *, time=-1, sigma=None, cmap="viridis", symmetric=False, elev=25, azim=-55, vmin=None, vmax=None):
    arr = get_field(field_name, time=time, sigma=sigma)
    if arr.ndim != 3:
        raise ValueError(f"{field_name} must resolve to (face, j, i), got shape={arr.shape}")
    if vmin is None or vmax is None:
        vmin_auto, vmax_auto = robust_limits(arr, symmetric=symmetric)
        vmin = vmin_auto if vmin is None else vmin
        vmax = vmax_auto if vmax is None else vmax
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    cm = plt.get_cmap(cmap)
    fig = plt.figure(figsize=(8, 8), constrained_layout=True)
    ax = fig.add_subplot(111, projection="3d")

    polygons = []
    polygon_values = np.array([])
    if corner_lon_name and corner_lat_name:
        polygons, polygon_values = cell_corner_polygons(
            arr,
            ds[corner_lon_name].values,
            ds[corner_lat_name].values,
        )

    if polygons:
        collection = Poly3DCollection(
            polygons,
            facecolors=cm(norm(polygon_values)),
            edgecolors=(0, 0, 0, 0.18),
            linewidths=0.05,
            antialiased=False,
            zsort="average",
        )
        ax.add_collection3d(collection)
        title_prefix = "Native cs32 cell polygons"
    else:
        lon = ds[lon_name].values
        lat = ds[lat_name].values
        x, y, z = lonlat_to_xyz(lon, lat)
        ax.scatter(
            x.ravel(), y.ravel(), z.ravel(),
            c=arr.ravel(), cmap=cm, norm=norm, s=8, linewidths=0, alpha=0.95,
        )
        title_prefix = "Native cs32 cell centres fallback"

    ax.view_init(elev=elev, azim=azim)
    configure_sphere_axes(ax)
    label = field_name if sigma is None else f"{field_name}, sigma index {sigma}"
    t_days = float(ds.time.isel(time=time)) / 86400 if "time" in ds else np.nan
    ax.set_title(f"{title_prefix}: {label}\nt = {t_days:.1f} days")
    mappable = plt.cm.ScalarMappable(norm=norm, cmap=cm)
    fig.colorbar(mappable, ax=ax, shrink=0.65, pad=0.02)
    return fig

## Coupled atmosphere/ocean snapshot

These are native six-face views. Each panel is one small `face, j, i` tile of
the cubed sphere, so compact rectangles here are expected. The atmosphere is shown with near-surface air
temperature and sea-ice fraction; the ocean is shown with SST and sea-surface
height. The values should look physically different but spatially coupled at
the ice/ocean/land boundaries.

In [ ]:
fields = [
    ("atm_TS", "AIM near-surface air T [K]", "magma", False),
    ("atm_SI_Fract", "AIM sea-ice fraction", "Blues", False),
    ("ocn_THETA", "Ocean SST", "magma", False),
    ("ocn_ETAN", "Ocean SSH anomaly", "RdBu_r", True),
]

fig, axes = plt.subplots(len(fields), ds.sizes["face"], figsize=(14, 8), constrained_layout=True)
for row, (name, label, cmap, symmetric) in enumerate(fields):
    if name not in ds:
        for ax in axes[row]:
            ax.axis("off")
        axes[row, 0].text(0.5, 0.5, f"missing {name}", ha="center", va="center")
        continue
    arr = get_field(name, time=-1)
    vmin, vmax = robust_limits(arr, symmetric=symmetric)
    for f, ax in enumerate(axes[row]):
        title = f"{label}\nface {f + 1}" if f == 0 else f"face {f + 1}"
        im = show_panel(ax, arr[f], title, cmap=cmap, vmin=vmin, vmax=vmax)
    fig.colorbar(im, ax=axes[row], shrink=0.7, location="right")
fig.suptitle(f"Coupled snapshot at t = {float(ds.time.isel(time=-1)) / 86400:.1f} days")
plt.show()

## Cube-unfolded views

Sadourny-style cross layout: faces 1-4 along the equator, face 5 above, face 6
below. This is not a map projection; it is a fast visual check for cube-face
continuity and face-specific artefacts.

In [ ]:
plot_cross("atm_TS", time=-1, cmap="magma")
plot_cross("ocn_ETAN", time=-1, cmap="RdBu_r", symmetric=True)
plt.show()

## AIM vertical structure

AIM has five sigma levels. The notebook detects the vertical coordinate name
from the opened dataset and shows potential temperature on each
level for one cube face, then a simple globally weighted mean profile. This is
useful for checking that the atmospheric column stays stably stratified.

In [ ]:
face = int(os.environ.get("FACE", "0"))
theta = ds["atm_THETA"].isel(time=-1)
vdim = require_vertical_dim(theta, "atm_THETA")
nlev = theta.sizes[vdim]

vmin, vmax = robust_limits(theta.values, symmetric=False)
fig, axes = plt.subplots(1, nlev, figsize=(14, 3), constrained_layout=True)
axes = np.atleast_1d(axes)
for k, ax in enumerate(axes):
    arr = theta.isel({vdim: k, "face": face}).values
    im = show_panel(ax, arr, f"sigma {k + 1}\nface {face + 1}", cmap="magma", vmin=vmin, vmax=vmax)
fig.colorbar(im, ax=axes, shrink=0.75, location="right")
fig.suptitle("AIM potential temperature by sigma level")
plt.show()

yc = ds[lat_name].values
weights = np.cos(np.deg2rad(yc))
weights = weights / np.nansum(weights)
theta_vals = theta.transpose(vdim, "face", "j", "i").values
profile = np.nansum(theta_vals * weights[None, ...], axis=(-3, -2, -1))

fig, ax = plt.subplots(figsize=(4, 5), constrained_layout=True)
ax.plot(profile, np.arange(1, len(profile) + 1), marker="o")
ax.invert_yaxis()
ax.set_xlabel("area-weighted mean THETA [K]")
ax.set_ylabel("sigma index (1 = top, 5 = surface)")
ax.set_title("AIM vertical mean profile")
ax.grid(True, alpha=0.3)
plt.show()

## Coupled time series

Area-weighted means use `cos(latitude)` as a lightweight proxy for cell area.
That is sufficient for spotting drift or blow-ups in quick inspection; exact
conservation checks should use MITgcm grid-cell areas.

In [ ]:
weights = np.cos(np.deg2rad(ds[lat_name].values))
weights = weights / np.nansum(weights)

def area_mean_2d(name):
    arr = ds[name].values  # (time, face, j, i)
    return np.nansum(arr * weights[None, ...], axis=(-3, -2, -1))

def area_mean_3d_surface(name):
    arr = select_vertical(ds[name], -1).values
    return np.nansum(arr * weights[None, ...], axis=(-3, -2, -1))

t_days = ds.time.values / 86400.0
ts_mean = area_mean_2d("atm_TS")
sst_mean = area_mean_2d("ocn_THETA")
ice_mean = area_mean_2d("atm_SI_Fract")
ssh_mean = area_mean_2d("ocn_ETAN")
atm_u = select_vertical(ds["atm_UVEL"], -1).values
atm_v = select_vertical(ds["atm_VVEL"], -1).values
atm_speed_max = np.sqrt(atm_u ** 2 + atm_v ** 2).max(axis=(-3, -2, -1))

fig, axes = plt.subplots(1, 4, figsize=(18, 4), constrained_layout=True)
axes[0].plot(t_days, ts_mean, label="atm_TS")
axes[0].plot(t_days, sst_mean, label="ocn_THETA")
axes[0].set_title("global mean temperature")
axes[0].legend()
axes[1].plot(t_days, ice_mean)
axes[1].set_title("mean sea-ice fraction")
axes[2].plot(t_days, ssh_mean)
axes[2].set_title("mean SSH anomaly")
axes[3].plot(t_days, atm_speed_max)
axes[3].set_title("max near-surface atm speed")
for ax in axes:
    ax.set_xlabel("time [days]")
    ax.grid(True, alpha=0.3)
plt.show()

## 3D native cubed-sphere rendering

This view uses `XG`/`YG` corner coordinates to draw each native cs32 cell as a
polygon on the unit sphere. It highlights the key peculiarity of the dataset:
the canonical data are not stored on a lat/lon raster, but on MITgcm's cubed
sphere. Face seams are expected; folded or rectangular patches are not.

In [ ]:
plot_sphere("atm_TS", time=-1, cmap="magma", elev=25, azim=-55)
plot_sphere("ocn_ETAN", time=-1, cmap="RdBu_r", symmetric=True, elev=20, azim=35)
plt.show()

## Animations

These MP4 cells follow the canvas-grab pattern used by the other visualization
notebooks. They keep fixed color limits across frames so visual changes reflect
model evolution rather than autoscaling. Set `ANIMATION_MAX_FRAMES` in the
environment to cap long runs during quick inspection.

In [ ]:
import imageio.v3 as iio
from IPython.display import Video, display
from tqdm import trange

ANIMATION_MAX_FRAMES = int(os.environ.get("ANIMATION_MAX_FRAMES", "48"))

def animation_indices(max_frames=ANIMATION_MAX_FRAMES):
    ntime = ds.sizes.get("time", 1)
    nframes = min(ntime, max_frames)
    return np.unique(np.linspace(0, ntime - 1, nframes, dtype=int))

def canvas_rgb(fig):
    fig.canvas.draw()
    w_px, h_px = fig.canvas.get_width_height()
    frame = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape(h_px, w_px, 4)
    return frame[: h_px - (h_px % 2), : w_px - (w_px % 2), :3]

def write_mp4(frames, output_path, *, fps=8):
    output_path = Path(output_path)
    iio.imwrite(
        str(output_path),
        np.stack(frames, axis=0),
        fps=fps,
        codec="libx264",
        pixelformat="yuv420p",
    )
    print(f"Wrote {output_path}  ({len(frames)} frames @ {fps} fps)")
    display(Video(str(output_path), embed=False))

def time_days(index):
    return float(ds.time.isel(time=int(index))) / 86400.0 if "time" in ds else float(index)

### Coupled native-face overview

Four synchronized rows show the atmosphere, ice, ocean temperature, and sea
surface height on all six native cubed-sphere faces. This is the fastest way
to see whether the coupled system evolves coherently across component and face
boundaries.

In [ ]:
fps = 8
dpi = 110
time_ids = animation_indices()
output_path = Path("cpl_aim_ocn_coupled_faces.mp4")

anim_fields = [
    ("atm_TS", "AIM near-surface air T [K]", "magma", False),
    ("atm_SI_Fract", "sea-ice fraction", "Blues", False),
    ("ocn_THETA", "ocean SST", "magma", False),
    ("ocn_ETAN", "ocean SSH anomaly", "RdBu_r", True),
]
anim_fields = [item for item in anim_fields if item[0] in ds]
limits = {
    name: robust_limits(ds[name].values, symmetric=symmetric)
    for name, _, _, symmetric in anim_fields
}

frames = []
for pos in trange(len(time_ids), desc="coupled faces"):
    ti = int(time_ids[pos])
    fig, axes = plt.subplots(len(anim_fields), ds.sizes["face"], figsize=(14, 2.2 * len(anim_fields)), constrained_layout=True, dpi=dpi)
    axes = np.atleast_2d(axes)
    for row, (name, label, cmap, _) in enumerate(anim_fields):
        arr = get_field(name, time=ti)
        vmin, vmax = limits[name]
        for face, ax in enumerate(axes[row]):
            title = f"{label}\nface {face + 1}" if face == 0 else f"face {face + 1}"
            im = show_panel(ax, arr[face], title, cmap=cmap, vmin=vmin, vmax=vmax)
        fig.colorbar(im, ax=axes[row], shrink=0.72, location="right")
    fig.suptitle(f"Coupled AIM + ocean native cs32 state | t = {time_days(ti):.1f} days")
    frames.append(canvas_rgb(fig))
    plt.close(fig)

write_mp4(frames, output_path, fps=fps)

### AIM vertical-column evolution

This animation keeps one cube face on screen and advances through time. The
left panels show potential temperature at every AIM sigma level; the profile on
the right tracks the global area-weighted vertical structure.

In [ ]:
fps = 8
dpi = 110
time_ids = animation_indices()
output_path = Path("cpl_aim_ocn_atm_theta_vertical.mp4")
face = int(os.environ.get("FACE", "0"))

theta_all = ds["atm_THETA"]
vdim = require_vertical_dim(theta_all, "atm_THETA")
nlev = theta_all.sizes[vdim]
vmin, vmax = robust_limits(theta_all.values, symmetric=False)
weights = np.cos(np.deg2rad(ds[lat_name].values))
weights = weights / np.nansum(weights)
theta_profiles = theta_all.transpose("time", vdim, "face", "j", "i").values
theta_profiles = np.nansum(theta_profiles * weights[None, None, ...], axis=(-3, -2, -1))
pmin, pmax = robust_limits(theta_profiles, symmetric=False)

frames = []
for pos in trange(len(time_ids), desc="AIM vertical"):
    ti = int(time_ids[pos])
    theta = theta_all.isel(time=ti)
    fig = plt.figure(figsize=(15, 4), constrained_layout=True, dpi=dpi)
    gs = fig.add_gridspec(1, nlev + 1, width_ratios=[1] * nlev + [1.25])
    level_axes = [fig.add_subplot(gs[0, k]) for k in range(nlev)]
    profile_ax = fig.add_subplot(gs[0, -1])

    for k, ax in enumerate(level_axes):
        arr = theta.isel({vdim: k, "face": face}).values
        im = show_panel(ax, arr, f"sigma {k + 1}\nface {face + 1}", cmap="magma", vmin=vmin, vmax=vmax)
    fig.colorbar(im, ax=level_axes, shrink=0.78, location="bottom", label="THETA [K]")

    profile = theta_profiles[ti]
    profile_ax.plot(theta_profiles.T, np.arange(1, nlev + 1), color="0.75", linewidth=0.7, alpha=0.35)
    profile_ax.plot(profile, np.arange(1, nlev + 1), marker="o", color="tab:red", linewidth=2)
    profile_ax.set_xlim(pmin, pmax)
    profile_ax.set_ylim(nlev + 0.3, 0.7)
    profile_ax.set_xlabel("global mean THETA [K]")
    profile_ax.set_ylabel("sigma index")
    profile_ax.grid(True, alpha=0.3)
    profile_ax.set_title("vertical profile")
    fig.suptitle(f"AIM potential temperature vertical evolution | t = {time_days(ti):.1f} days")
    frames.append(canvas_rgb(fig))
    plt.close(fig)

write_mp4(frames, output_path, fps=fps)

### Rotating native-sphere animation

The spherical movie uses the same native cell-polygon renderer as the static 3D
view and slowly rotates the camera. Override the variable with `SPHERE_FIELD`,
for example `SPHERE_FIELD=ocn_ETAN`.

In [ ]:
fps = 8
dpi = 110
time_ids = animation_indices(max_frames=min(ANIMATION_MAX_FRAMES, 36))
sphere_field = os.environ.get("SPHERE_FIELD", "atm_TS")
sphere_cmap = "RdBu_r" if sphere_field.endswith(("ETAN", "UVEL", "VVEL")) else "magma"
sphere_symmetric = sphere_cmap == "RdBu_r"
output_path = Path(f"cpl_aim_ocn_{sphere_field}_sphere.mp4")

sphere_values = np.stack([get_field(sphere_field, time=int(ti)) for ti in time_ids], axis=0)
vmin, vmax = robust_limits(sphere_values, symmetric=sphere_symmetric)

frames = []
for pos in trange(len(time_ids), desc="native sphere"):
    ti = int(time_ids[pos])
    azim = -55 + 360.0 * pos / max(1, len(time_ids))
    fig = plot_sphere(
        sphere_field,
        time=ti,
        cmap=sphere_cmap,
        symmetric=sphere_symmetric,
        elev=22,
        azim=azim,
        vmin=vmin,
        vmax=vmax,
    )
    fig.set_dpi(dpi)
    frames.append(canvas_rgb(fig))
    plt.close(fig)

write_mp4(frames, output_path, fps=fps)

## Quick sanity checks

These checks are intentionally coarse. They are meant to flag obviously
unphysical smoke/preflight outputs before launching or trusting a long sweep.

In [ ]:
checks = {
    "atm_TS_min_K": float(ds["atm_TS"].min()),
    "atm_TS_max_K": float(ds["atm_TS"].max()),
    "atm_THETA_min_K": float(ds["atm_THETA"].min()),
    "atm_THETA_max_K": float(ds["atm_THETA"].max()),
    "ocn_THETA_min": float(ds["ocn_THETA"].min()),
    "ocn_THETA_max": float(ds["ocn_THETA"].max()),
    "sea_ice_fraction_min": float(ds["atm_SI_Fract"].min()),
    "sea_ice_fraction_max": float(ds["atm_SI_Fract"].max()),
}
for key, value in checks.items():
    print(f"{key:24s} {value:12.5g}")